In [ ]:
# Local execution requires an NVIDIA GPU with at least 12 GB VRAM.
# For cloud execution, Google Colab with a GPU runtime can be used.

## Base library Installation
# Install Baselines for model comparison
!uv pip install catboost xgboost

# Install the datasets library for loading example data
!uv pip install datasets

# Install rich for better and more readable printing
!uv pip install rich

## TabPFN Installation optimized for Google Colab
# Install the TabPFN Client library
!uv pip install tabpfn-client

# Install TabPFN extensions for additional functionalities
!uv pip install 'tabpfn-extensions[all]'

# Install tabpfn
!uv pip install tabpfn

In [ ]:
# Authentication code - No need to read it!
import os
import getpass
from rich.console import Console
from rich.rule import Rule
from rich.panel import Panel

console = Console()
# --- Prior Labs Authentication ---
console.print(Rule("[bold]Prior Labs Authentication[/bold]"))
console.print(
    "\nTabPFN model weights or API access via client requires a free "
    "[bold]Prior Labs account[/bold] and acceptance of the non-commercial "
    "license.\n"
)

tabpfn_token = None

# 1. Try Colab secret TABPFN_TOKEN
try:
    from google.colab import userdata
    tabpfn_token = userdata.get("TABPFN_TOKEN")
    if tabpfn_token:
        os.environ["TABPFN_TOKEN"] = tabpfn_token
        console.print("[bold green]✅ Found TABPFN_TOKEN in Colab secrets.[/bold green]")
except Exception:
    pass

# 2. If no token found, prompt the user
if not tabpfn_token:
    console.print(
        Panel(
            "To get your access token:\n\n"
            "  1. Go to [link=https://ux.priorlabs.ai]ux.priorlabs.ai[/link] and sign up / log in\n"
            "  2. Accept the license at [link=https://ux.priorlabs.ai/account/licenses]ux.priorlabs.ai/account/licenses[/link]\n"
            "  3. Copy your Access Token from [link=https://ux.priorlabs.ai/account]ux.priorlabs.ai/account[/link]\n\n"
            "[bold yellow]Tip:[/bold yellow] Save the token as a Colab secret named "
            "[bold cyan]TABPFN_TOKEN[/bold cyan] to skip this step next time.",
            title="[bold]🔑 Prior Labs Access Token required",
            border_style="blue",
        )
    )
    while not tabpfn_token:
        token_input = getpass.getpass("Paste your TABPFN_TOKEN and press Enter: ")
        if token_input.strip():
            tabpfn_token = token_input.strip()
            os.environ["TABPFN_TOKEN"] = tabpfn_token
        else:
            console.print("[red]Token cannot be empty. Please try again.[/red]")



console.print(Panel.fit("[bold magenta]TabPFN Demo: Backend Selection[/bold magenta]"))
console.print("\nThis script can run TabPFN using one of two backends:")
console.print("  [bold]local:[/bold] Uses a local GPU (NVIDIA). Requires CUDA.")
console.print(
    "  [bold]client:[/bold] Uses the TabPFN API. Requires an internet connection and a free account."
)

backend = None
while backend is None:
  console.print(
      "\n[bold]Choose your backend[/bold]: - If no text box is shown, restart the cell.",
  )
  user_input = input("Enter 'local' or 'client' and press return:")
  if user_input not in ["local", "client"]:
    continue
  backend = user_input

console.print(
    f"\n✅ You have selected the '[bold green]{backend}[/bold green]' backend."
)

console.print(Rule(f"[bold]Setting up [cyan]{backend}[/cyan] backend[/bold]"))

if backend == "local":
    console.print("Attempting local backend setup...")
    import torch

    if not torch.cuda.is_available():
        console.print(
            "[bold red]Error:[/bold red] GPU device not found. For fast training, please enable GPU.",
            style="red",
        )
        console.print(
            "In Colab: Go to [bold]Runtime -> Change runtime type -> Hardware accelerator -> GPU.[/bold]",
            style="yellow",
        )
        raise SystemError("GPU device not found.")
    console.print("[bold green]✅ GPU is available.[/bold green]")




    console.print("")
    console.print("Importing local TabPFN library...")

    from tabpfn import TabPFNClassifier, TabPFNRegressor

    console.print("[bold green]✅ TabPFN (local) imported successfully.[/bold green]")
elif backend == "client":
    console.print("Attempting client backend setup...")
    console.print("Importing TabPFN client library...")
    from tabpfn_client import TabPFNClassifier, TabPFNRegressor, init

    init()
    console.print("[bold green]✅ TabPFN (client) initialized.[/bold green]")

In [ ]:
import os
from pathlib import Path
import torch
import pandas as pd
from tabpfn import TabPFNClassifier
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report
)

if 'tabpfn_token' in globals() and tabpfn_token:
    os.environ["TABPFN_TOKEN"] = tabpfn_token

data_dir = Path(os.environ.get("DATA_DIR", "../data"))
if not (data_dir / "X_train.csv").exists():
    data_dir = Path("data")

X_train = pd.read_csv(data_dir / "X_train.csv")
y_train = pd.read_csv(data_dir / "y_train.csv").squeeze("columns")

X_test = pd.read_csv(data_dir / "X_test.csv")
y_test = pd.read_csv(data_dir / "y_test.csv").squeeze("columns")

device = "cuda" if torch.cuda.is_available() else "cpu"

model = TabPFNClassifier(
    device=device,
    random_state=42
)

model.fit(X_train, y_train)

y_pred = model.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)
f1_macro = f1_score(y_test, y_pred, average="macro")
f1_weighted = f1_score(y_test, y_pred, average="weighted")

print(f"Device: {device}")
print(f"Accuracy:    {accuracy:.4f}")
print(f"Macro F1:    {f1_macro:.4f}")
print(f"Weighted F1: {f1_weighted:.4f}")

# Classification report: 26 classes
print("\nClassification Report:")
print(classification_report(
    y_test,
    y_pred,
    labels=list(range(26)),
    digits=4,
    zero_division=0
))

In [ ]:
import os
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix

# 1. Output directory (configurable via RESULTS_DIR or relative path)
default_cm_dir = Path("../experiment/results/subpop/confusion_matrix")
if not default_cm_dir.parent.exists():
    default_cm_dir = Path("experiment/results/subpop/confusion_matrix")
cm_dir = Path(os.environ.get("RESULTS_DIR", default_cm_dir))
cm_dir.mkdir(parents=True, exist_ok=True)

# 3. Normalized confusion matrix
labels = list(range(26))

cm_norm_true = confusion_matrix(
    y_test,
    y_pred,
    labels=labels,
    normalize="true"
)

# Transpose: X = Actual, Y = Predicted
cm_plot = cm_norm_true.T

# 4. Plot
x = np.arange(26)

fig, ax = plt.subplots(figsize=(14, 12), dpi=150)

im = ax.imshow(
    cm_plot,
    interpolation="nearest",
    cmap="RdBu_r",
    vmin=0.0,
    vmax=1.0
)

# Colorbar
cbar = fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
cbar.set_label(
    "Prediction Ratio (0.0: Dark Blue -> 1.0: Dark Red)",
    fontsize=10
)

# Axis
ax.set_xticks(x)
ax.set_yticks(x)

ax.set_xticklabels(labels, rotation=45, ha="right", fontweight="bold")
ax.set_yticklabels(labels, fontweight="bold")

ax.set_xlabel("Actual Label", fontsize=12, fontweight="bold", labelpad=10)
ax.set_ylabel("Predicted Label", fontsize=12, fontweight="bold", labelpad=10)

ax.set_title(
    "TabPFN - Normalized Confusion Matrix",
    fontsize=14,
    fontweight="bold",
    pad=15
)

# 5. Annotations: only one value per cell
for i in range(26):
    for j in range(26):
        val = cm_plot[i, j]

        if val > 0.005 or i == j:
            color = "white" if (val < 0.25 or val > 0.75) else "black"

            ax.text(
                j, i,
                f"{val:.2f}",
                ha="center",
                va="center",
                color=color,
                fontsize=7,
                fontweight="bold" if i == j else "normal"
            )

plt.tight_layout()

# 6. Save
cm_path = os.path.join(cm_dir, "tabpfn_confusion_matrix.png")

plt.savefig(
    cm_path,
    dpi=300,
    bbox_inches="tight"
)

plt.show()
plt.close()

print(f"Saved Confusion Matrix: {cm_path}")